# Clinical Data Repository — Governance Overview

**Project:** RMIT Clinical Data Repository Capstone  
**Platform:** Databricks on AWS  
**Primary Dataset:** MIMIC-IV  
**Governance Platform:** Databricks Unity Catalog  

---

## 1. Purpose

This notebook documents the governance framework used for the Clinical Data Repository (CDR).

The project uses **Databricks Unity Catalog** as the central governance layer across the Medallion Architecture.

The governance framework is designed to support:

- Centralised data discovery
- Consistent metadata management
- Role-based access control
- Least-privilege access
- Clinical data classification
- Column-level protection
- Row-level security
- Data-quality monitoring
- Data lineage
- Auditability
- Governed analytics and AI consumption

The objective is to create a Clinical Data Repository where data can move from raw ingestion to trusted analytical consumption while remaining **discoverable, controlled, traceable, and reusable**.

> **This notebook is descriptive only. It does not modify permissions, tables, masking policies, row filters, or source data.**

---

# 2. Databricks Governance Architecture

Databricks Unity Catalog provides the central governance layer for the Clinical Data Repository.

Unity Catalog organises governed assets using a hierarchical namespace:

```text
Metastore
    │
    └── Catalog
          │
          └── Schema
                │
                ├── Table
                ├── View
                └── Volume
```

The primary catalog used by the project is:

```text
cdr_mimiciv
```

Within the catalog, the Clinical Data Repository follows the Medallion Architecture:

```text
cdr_mimiciv
│
├── bronze
├── silver
└── gold
```

This provides clear separation between:

- source-aligned data,
- transformed and quality-controlled data,
- and analytics-ready data.

Unity Catalog provides the governance layer across all three stages.

---

# 3. Governed Source Architecture

The primary clinical dataset used by the project is **MIMIC-IV**.

The source files are stored in AWS S3 and made available to Databricks through a **Unity Catalog external volume**.

The source architecture is:

```text
MIMIC-IV
    │
    ▼
AWS S3
    │
    ▼
Unity Catalog External Volume
    │
    ▼
/Volumes/cdr_mimiciv/bronze/s3_mimic_iv
    │
    ├── hosp/
    └── icu/
```

Unity Catalog Volumes allow non-tabular source files, such as compressed CSV files, to participate in the same governed namespace as tables and views.

This means governance can begin at the source-data boundary rather than only after data has been transformed into tables.

---

# 4. Medallion Governance Model

The Clinical Data Repository follows the Databricks Medallion Architecture.

```text
Source Data
     │
     ▼
┌─────────┐
│ BRONZE  │
└────┬────┘
     │
     ▼
┌─────────┐
│ SILVER  │
└────┬────┘
     │
     ▼
┌─────────┐
│  GOLD   │
└────┬────┘
     │
     ▼
Analytics / Research / AI
```

Each layer has a different purpose and governance responsibility.

---

## 4.1 Bronze — Source-Aligned Layer

The Bronze layer represents the ingestion boundary of the Clinical Data Repository.

Its primary objective is to preserve the source structure while converting source files into governed Delta tables.

Bronze remains close to the original MIMIC-IV source data.

Typical Bronze responsibilities include:

- Source ingestion
- Schema capture
- Delta storage
- Source-level validation
- Controlled access to raw clinical data
- Traceability back to source files

Bronze should avoid research-specific transformations or analytical assumptions.

```text
MIMIC-IV Source Files
        │
        ▼
      Bronze
        │
        ▼
Source-Aligned Delta Tables
```

---

## 4.2 Silver — Trusted and Conformed Layer

The Silver layer transforms source-aligned data into reusable and quality-controlled clinical datasets.

Silver is designed to support multiple downstream analytical use cases rather than one specific research question.

Typical Silver responsibilities include:

- Data-type standardisation
- Structural validation
- Relationship validation
- Clinical semantic documentation
- Reusable data-quality checks
- Data-quality flags
- Consistent transformation logic

The Silver layer retains clinically meaningful source information while making quality issues visible.

Unusual or questionable clinical records should not automatically be deleted simply because they appear unexpected.

Where appropriate, quality conditions can instead be represented using explicit data-quality indicators.

```text
Bronze
   │
   ▼
Silver
   │
   ├── Standardised
   ├── Validated
   ├── Conformed
   └── Quality Controlled
```

---

## 4.3 Gold — Analytics and Consumption Layer

The Gold layer provides analytics-ready datasets designed for downstream consumers.

Gold may contain:

- Analytical dimensions
- Analytical facts
- Derived measures
- Cohort logic
- Business definitions
- Research definitions
- Aggregations
- Dashboard-ready datasets

Typical Gold consumers include:

- Researchers
- Clinical leadership
- BI developers
- AI/BI dashboards
- Databricks Genie
- Analytical applications

```text
Silver
   │
   ▼
Gold
   │
   ├── Research
   ├── Dashboards
   ├── AI/BI
   ├── Genie
   └── Analytics
```

Research-specific or analytical definitions belong primarily in Gold rather than being embedded into reusable Silver datasets.

---

# 5. Unity Catalog as the Governance Foundation

Unity Catalog provides a central governance layer across the Clinical Data Repository.

Instead of managing governance separately for every individual dataset or storage location, Unity Catalog provides a common framework for:

- Cataloguing data
- Discovering data
- Managing permissions
- Applying metadata
- Managing tags
- Protecting columns
- Applying row-level controls
- Tracking lineage
- Supporting auditability

Conceptually:

```text
                    UNITY CATALOG
                          │
          ┌───────────────┼───────────────┐
          │               │               │
          ▼               ▼               ▼
       Metadata        Security       Observability
          │               │               │
       Catalog            RBAC           Lineage
       Tags               Masks          Auditing
       Comments           Filters        Quality
          │               │               │
          └───────────────┼───────────────┘
                          │
                          ▼
                     MIMIC-IV CDR
```

---

# 6. Governance Metadata Model

The Clinical Data Repository uses governed metadata to describe the purpose and sensitivity of data assets.

Six governed tags are applied (`04_Governance/tags`, executed by `jobs/01_apply_tags`):

| Tag | Level | Purpose |
|---|---|---|
| `layer` | schema, table | Identifies the Medallion layer of the asset |
| `type` | table | `fact`, `dimension` or `view` |
| `mimic_module` | table | Source module (`hosp` or `icu`) the data derives from |
| `domain` | table | Identifies the clinical subject area represented by the table |
| `hipaa_identifiability` | column | Classifies identifier or quasi-identifier sensitivity |
| `hl7_confidentiality` | column | Describes confidentiality requirements for clinical information |

The important distinction is:

> **Tags classify data. Policies determine how users are allowed to interact with that classified data.**

---

# 7. Layer Classification

The `layer` tag identifies the architectural stage of an asset.

Example values:

```text
bronze
silver
gold
```

| Layer | Meaning |
|---|---|
| `bronze` | Source-aligned ingestion data |
| `silver` | Trusted and reusable transformed data |
| `gold` | Analytics-ready consumption data |

The layer classification helps communicate the intended purpose of a dataset and supports access-control decisions.

---

# 8. Clinical Domain Classification

The `domain` tag categorises tables according to the clinical subject area they primarily represent.

The project uses an **OMOP-inspired domain classification** to provide consistent terminology across clinical datasets.

Example values include:

```text
person
visit
condition
procedure
drug
measurement
observation
location
care_site
provider
metadata
```

Examples include:

```text
patients
→ person

admissions
→ visit

diagnoses_icd
→ condition

procedures_icd
→ procedure

prescriptions
→ drug

labevents
→ measurement

chartevents
→ observation
```

The domain classification supports:

- Data discovery
- Clinical interoperability
- Access-policy design
- Research dataset identification
- Future cross-institutional use cases

> This classification does not mean that the MIMIC-IV tables have been converted into the OMOP Common Data Model. It provides a consistent clinical-domain classification for governance purposes.

---

# 9. Sensitive Data Classification

The governance framework also considers the sensitivity of individual data elements.

Two metadata dimensions are used for this purpose:

```text
hipaa_identifiability
```

and

```text
hl7_confidentiality
```

These classifications can support policies such as:

- Dynamic column masking
- Restricted-column access
- Row-level controls
- Researcher access restrictions
- Clinical-consumer access rules

MIMIC-IV is a de-identified clinical research dataset.

The controls implemented in this project therefore demonstrate how similar governance patterns could be applied in a production healthcare environment without claiming that MIMIC-IV identifiers represent directly identifiable live patients.

---

# 10. Governance Personas

Access should be assigned to groups or logical personas rather than directly to individual users wherever possible.

The CDR role catalog (`04_Governance/rbac/role_catalog.json`) defines seven groups, one for
each control the platform has to demonstrate:

| Group | Control demonstrated | Primary Responsibility |
|---|---|---|
| `cdr_admins` | Full administration | IT / eResearch / platform governance; SELECT + MODIFY on all layers, unmasked |
| `cdr_data_engineers` | Platform engineering | Pipeline engineering; read/write/build on all layers, unmasked where operationally required |
| `cdr_data_scientist` | User-level separation | ML / data science team; Silver SELECT only, direct + quasi identifiers masked |
| `cdr_standard_researcher` | Column level | Approved research; identifiers arrive as NULL |
| `cdr_researcher_psychiatry` | Row level | Same grants as the standard researcher, plus PSYCH rows |
| `cdr_pharmacy_analyst` | Table level | Pharmacy-related Gold tables only |
| `cdr_external_researcher` | Table level, narrow | Approved diagnostic tables only, direct identifiers masked |

`roles.pdf` specified eleven groups. The five that remain dropped each duplicated a control
above, and `03_create_groups` never revokes, so any grants a dropped group already holds
survive the edit and have to be revoked in the account console. `cdr_mortality_investigator`
was removed in the 2026-10-28 finalisation: HL7 Very Restricted data is now readable only by
the platform roles.

The catalog is the single source of truth: provisioning (`jobs/03_create_groups`),
verification (`jobs/04_verify_rbac`) and fine-grained controls
(`jobs/05_fine_grained_controls`) all read it rather than hard-coding permissions.

---

# 11. Layer-Based Access Model

The Medallion layers provide the first level of access separation.

| Group | Bronze | Silver | Gold | RLS level |
|---|---|---|---|---|
| `cdr_admins` | SELECT + MODIFY | SELECT + MODIFY | SELECT + MODIFY | Level 3 (Global) |
| `cdr_data_engineers` | SELECT + MODIFY | SELECT + MODIFY | SELECT + MODIFY | Level 3 (Global) |
| `cdr_data_scientist` | No access | SELECT | No access | Level 1 (Standard) |
| `cdr_researcher_psychiatry` | No access | No access | ALL domains | Level 2 (Standard + Psychiatry) |
| `cdr_standard_researcher` | No access | No access | ALL domains | Level 1 (Standard) |
| `cdr_pharmacy_analyst` | No access | No access | pharmacy | Level 1 (Standard) |
| `cdr_external_researcher` | No access | No access | diagnostics | Level 1 (Standard) |

The objective is to follow the principle of:

> **Least Privilege**

Users should receive only the level of access required to perform their responsibilities.

---

# 12. Role-Based and Attribute-Based Governance

The governance framework combines **Role-Based Access Control (RBAC)** with metadata-driven governance.

## Role-Based Access Control

RBAC determines access according to the user's organisational responsibility.

For example:

```text
cdr_admins
→ Bronze + Silver + Gold, SELECT + MODIFY      (schema-level control)

cdr_standard_researcher
→ whole Gold schema, identifiers masked        (column-level control)

cdr_researcher_psychiatry
→ same as cdr_standard_researcher, plus PSYCH rows  (row-level control)

cdr_data_engineers
→ Bronze + Silver + Gold, read/write/build      (platform engineering)

cdr_data_scientist
→ whole Silver schema, identifiers masked      (user-level separation)

cdr_pharmacy_analyst
→ Gold pharmacy tables only      (table-level control)

cdr_external_researcher
→ Gold diagnostics tables only, identifiers masked  (narrow table-level control)
```

## Attribute-Based Governance

Metadata attached to data assets can provide additional context when determining access.

Conceptually:

```text
User Persona
      +
Layer
      +
Clinical Domain
      +
Identifiability Classification
      +
Confidentiality Classification
      │
      ▼
Governance Policy
      │
      ▼
Allowed / Restricted / Masked
```

This provides a more scalable governance model than creating separate physical copies of datasets for every type of user.

---

# 13. Column-Level Protection

Some users may require access to a table without requiring access to every attribute in that table.

Dynamic column masking can provide different representations of the same governed column depending on the user's permissions.

Conceptually:

```text
                SAME GOVERNED TABLE
                       │
             ┌─────────┴─────────┐
             │                   │
     Authorised User      Restricted User
             │                   │
             ▼                   ▼
      Original Value        Masked Value
```

This allows sensitive fields to be protected without maintaining multiple physical copies of the same dataset.

Column masking and row filters are implemented in:

```text
04_Governance/jobs/05_fine_grained_controls.py   Gold and Silver tags -> masks; Gold RLS levels
04_Governance/jobs/01_apply_tags.py              Tags the Silver identifier columns that 05 masks
```

---

# 14. Row-Level Security

Row-level security provides an additional level of access control.

Instead of creating separate tables for different users, row filters can determine which records a user is allowed to retrieve.

Conceptually:

```text
             ONE GOVERNED TABLE
                    │
          ┌─────────┴─────────┐
          │                   │
       Group A             Group B
          │                   │
          ▼                   ▼
   Approved Rows        Approved Rows
```

This supports fine-grained access while maintaining a single governed source of truth.

---

# 15. Data Quality Governance

Governance is not limited to permissions.

A governed data platform must also communicate whether data is trustworthy and where quality issues exist.

The project therefore treats data quality as part of the governance framework.

The general approach is:

```text
Source Data
     │
     ▼
Silver Transformation
     │
     ▼
Data Quality Rules
     │
     ▼
Quality Indicators
     │
     ▼
Governance Reporting
```

Where clinically unusual records are encountered, the project prefers transparent quality indicators over silently deleting or modifying records without clear justification.

The dedicated data-quality governance notebook will summarise these controls across the Clinical Data Repository.

---

# 16. Centralised Data Discovery

Unity Catalog provides a central place for users to discover and understand data assets.

Through Catalog Explorer, users can inspect:

- Catalogs
- Schemas
- Tables
- Views
- Volumes
- Columns
- Data types
- Comments
- Tags
- Permissions
- Lineage

This reduces reliance on undocumented knowledge about where data is stored or which dataset should be used.

```text
User
 │
 ▼
Catalog Explorer
 │
 ├── What data exists?
 ├── What does it mean?
 ├── Which layer is it in?
 ├── Which clinical domain does it represent?
 ├── Who can access it?
 └── Where did it come from?
```

---

# 17. Data Lineage

Unity Catalog provides native lineage capabilities for supported Databricks operations.

The expected end-to-end lineage for the Clinical Data Repository is:

```text
AWS S3
   │
   ▼
Bronze
   │
   ▼
Silver
   │
   ▼
Gold
   │
   ├── AI/BI Dashboards
   │
   └── Databricks Genie
```

Rather than recreating a lineage system in custom code, the project uses the native Databricks lineage interface as part of the governance demonstration.

This allows users to visually inspect upstream and downstream dependencies.

---

# 18. Data Auditing

Governance also requires accountability.

Databricks provides native auditing and system information that can be used to understand platform activity.

Audit information can support questions such as:

```text
Who accessed the platform?

What action was performed?

Which object was accessed?

When did the activity occur?
```

The project uses Databricks-native auditing capabilities rather than recreating an audit platform through custom notebooks.

---

# 19. Governed Analytics and AI

The final objective of governance is not simply to restrict access.

Governance should enable trusted data to be safely used by downstream consumers.

The governed data path is:

```text
MIMIC-IV
    │
    ▼
Bronze
    │
    ▼
Silver
    │
    ▼
Gold
    │
    ├───────────────┐
    ▼               ▼
AI/BI Dashboard    Genie
    │               │
    └───────┬───────┘
            ▼
      Governed Insights
```

This helps ensure that analytics and AI applications operate on approved datasets rather than directly accessing uncontrolled source files.

---

# 20. Databricks-Native Governance Capabilities

The project deliberately uses Databricks-native functionality where the platform already provides the required governance capability.

| Governance Capability | Databricks Capability |
|---|---|
| Centralised governance | Unity Catalog |
| Data discovery | Catalog Explorer |
| Metadata | Comments and Tags |
| Access control | Unity Catalog Permissions |
| Column protection | Dynamic Column Masking |
| Row protection | Row Filters |
| Data lineage | Unity Catalog Lineage |
| Auditing | Databricks System / Audit Capabilities |
| Data quality | Silver Quality Rules + Governance Monitoring |
| Analytics | Databricks AI/BI |
| Natural-language analytics | Databricks Genie |

The governance notebooks therefore focus on implementing and validating project-specific governance rules rather than rebuilding functionality that Databricks already provides natively.

---

# 21. Avoiding Data Duplication

One objective of the governance model is to avoid unnecessary copies of the same clinical data.

Without central governance, organisations may create separate datasets for:

```text
Researchers
Clinical Leadership
Service-line researchers
BI Developers
External collaborators
```

This can lead to:

- Duplicate storage
- Inconsistent definitions
- Additional maintenance
- Difficult access management
- Multiple versions of the truth

The preferred approach is:

```text
              ONE GOVERNED DATASET
                       │
        ┌──────────────┼──────────────┐
        │              │              │
        ▼              ▼              ▼
 cdr_standard_    cdr_researcher_   cdr_data_
 researcher        psychiatry        scientist
        │              │              │
 Masked IDs +    Masked IDs +    Masked IDs,
 level 1 rows    level 2 rows    Silver only
```

Different consumers can therefore interact with the same governed data while receiving different permissions where required.

---

# 22. Future Enterprise Extensions

The current project focuses on building and governing the Clinical Data Repository.

The architecture can later be extended to additional Databricks capabilities.

## Lakehouse Federation

Future enterprise implementations could use Lakehouse Federation to govern and query supported external data systems without requiring every dataset to be physically migrated into the lakehouse.

Conceptually:

```text
External Database
       │
       ▼
Lakehouse Federation
       │
       ▼
Unity Catalog
       │
       ▼
Governed Query Access
```

This is a future-state capability and is not claimed as implemented within the current prototype.

---

## ML and Model Governance

The governance framework can also be extended from data assets to machine-learning assets.

A future architecture could include:

```text
Governed Silver / Gold Data
          │
          ▼
     ML Features
          │
          ▼
    ML Experiments
          │
          ▼
      ML Models
          │
          ▼
   Governed Deployment
```

Databricks MLflow and related model-management capabilities could support this extension.

The current project does not claim to implement a production ML governance framework.

---

## External Data Sharing

Where cross-organisation collaboration is required, governed sharing mechanisms can be introduced without manually exporting uncontrolled copies of data.

This may include controlled data-sharing technologies such as Delta Sharing.

External sharing is considered a future enterprise extension rather than a core requirement of the current prototype.

---

# 23. Governance Notebook Structure

The governance implementation lives in this folder and in `jobs/`.

| Notebook | Purpose |
|---|---|
| `01_Governance_Overview` | Documents the governance architecture, principles, and Databricks capabilities |
| `02_Catalog_Metadata_and_Tags` | Applies official MIMIC-IV comments to Bronze, Silver and Gold |
| `05_Data_Quality_Governance` | Measures and reports data-quality controls |
| `_archive/apply_masks_notebook` | Retired 2026-10-26; Silver masks now come from `jobs/01` + `jobs/05` |
| `rbac/role_catalog.json` | Single source of truth for groups, domains, masking and RLS |
| `jobs/01_apply_tags` | Applies the schema, table and column tag taxonomy |
| `jobs/02_verify_tags` | Fails the job when tags are missing or wrong |
| `jobs/03_create_groups` | Creates groups and least-privilege grants from the catalog |
| `jobs/04_verify_rbac` | Diffs live grants, masks and row filters against the catalog |
| `jobs/05_fine_grained_controls` | Row filters for RLS levels and column masks from the tags |
| `tags/` | Original standalone tagging scripts and taxonomy reference |
| `test/governance_test_notebook` | Manual identity test: what each role can actually see |

Native Databricks capabilities such as lineage and auditing are demonstrated directly through the Databricks platform rather than recreated as separate notebooks.

---

# 24. Governance Framework Summary

The overall governance architecture is:

```text
                         UNITY CATALOG
                               │
         ┌─────────────────────┼─────────────────────┐
         │                     │                     │
         ▼                     ▼                     ▼
   DATA DISCOVERY        ACCESS CONTROL        OBSERVABILITY
         │                     │                     │
      Catalog                  RBAC                Lineage
      Metadata                 ABAC                Auditing
      Tags                     Masks               Quality
      Comments                 Filters
         │                     │                     │
         └─────────────────────┼─────────────────────┘
                               │
                               ▼
                         CDR MEDALLION
                               │
                ┌──────────────┼──────────────┐
                ▼              ▼              ▼
             BRONZE          SILVER          GOLD
                │              │              │
          Data Engineers   Data Engineers  Data Engineers
          Admins           Researchers     Researchers
                           (service line)  (standard)
                               │
                               ▼
                      GOVERNED CONSUMPTION
                               │
                     ┌─────────┴─────────┐
                     ▼                   ▼
                 AI/BI Dashboard       Genie
```

---

# 25. Governance Principle

The governance framework can be summarised as:

> **One governed clinical data platform, one trusted path through the Medallion Architecture, and controlled access based on user responsibility and data classification.**

**Unity Catalog** provides the governance foundation.

**Bronze** provides source-aligned data.

**Silver** provides trusted and reusable clinical data.

**Gold** provides analytics-ready information.

Databricks security, metadata, lineage, auditing, AI/BI, and Genie capabilities provide the governed platform experience around those datasets.